# Load the  dataset from online

Load roughly 858MB dataset of [simulated financial transactions from Kaggle](https://www.kaggle.com/datasets/conorsully1/simulated-transactions)

In [1]:
# !wget https://storage.googleapis.com/rapidsai/polars-demo/transactions-t4-20.parquet -O transactions.parquet

# Create the DuckDB database

In [2]:
import duckdb

# Paths
db_path = 'transactions.duckdb'
parquet_path = 'transactions.parquet'

# Connect to DuckDB
conn = duckdb.connect(db_path)

# Load Parquet directly into a table
conn.execute(f"""
CREATE OR REPLACE TABLE trxns AS 
SELECT * FROM read_parquet('{parquet_path}');
""")

# Setup Multi-GPU Engine

In [3]:
# This will use all GPUs on the local host by default
from dask_cuda import LocalCUDACluster
from dask.distributed import Client, wait
import polars as pl

client = Client(LocalCUDACluster())
# client = Client(LocalCUDACluster(n_workers=2))  # to specify a specific number of workers
client

/home/allisond/miniconda3/envs/rapids-25.06/lib/python3.10/site-packages/optuna/study/_optimize.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module


Connection method: Cluster object,Cluster type: dask_cuda.LocalCUDACluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 62.61 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43769,Workers: 2
Dashboard: http://127.0.0.1:8787/status,Total threads: 2
Started: Just now,Total memory: 62.61 GiB
Comm: tcp://127.0.0.1:35789,Total threads: 1
Dashboard: http://127.0.0.1:41433/status,Memory: 31.31 GiB
Nanny: tcp://127.0.0.1:34705,


In [4]:
executor_options = {"scheduler": "distributed"}  # Use "synchronous" for single GPU streaming execution
executor = "streaming"

engine_multi = pl.GPUEngine(
    executor=executor,
    executor_options=executor_options,
)

# Pull the data using GPU Polars

Pull the data from the DuckDB database 

In [5]:
import polars as pl

pl_trxns = pl.from_arrow(
    duckdb.connect("transactions.duckdb")
           .execute("SELECT * FROM trxns")
           .arrow()
).lazy()

In [6]:
type(pl_trxns)

polars.lazyframe.frame.LazyFrame

Observe the first 5 rows of the data

In [7]:
(
    pl_trxns
    .head()
    .collect(engine = engine_multi)
)

CUST_ID,START_DATE,END_DATE,TRANS_ID,DATE,YEAR,MONTH,DAY,EXP_TYPE,AMOUNT
str,date,date,str,date,i64,i64,i64,str,f64
"""CI6XLYUMQK""",2015-05-01,null,"""T8I9ZB5A6X90UG8""",2015-09-11,2015,9,11,"""Motor/Travel""",20.27
"""CI6XLYUMQK""",2015-05-01,null,"""TZ4JSLS7SC7FO9H""",2017-02-08,2017,2,8,"""Motor/Travel""",12.85
"""CI6XLYUMQK""",2015-05-01,null,"""TTUKRDDJ6B6F42H""",2015-08-01,2015,8,1,"""Housing""",383.8
"""CI6XLYUMQK""",2015-05-01,null,"""TDUHFRUKGPPI6HD""",2019-03-16,2019,3,16,"""Entertainment""",5.72
"""CI6XLYUMQK""",2015-05-01,null,"""T0JBZHBMSVRFMMD""",2015-05-15,2015,5,15,"""Entertainment""",11.06


# Filter and group transactions by year, month, and type, then compute total, average, and count of amounts, returning only high-value groups

In [8]:
%%time

result = (
    pl_trxns
    .head(500_000)
    .filter(
        pl.col("YEAR").is_in([2015, 2017, 2019]) &
        pl.col("EXP_TYPE").is_in(["Motor/Travel", "Entertainment", "Housing"]) &
        (pl.col("AMOUNT") > 300)
    )
    .group_by(["YEAR", "MONTH", "EXP_TYPE"])
    .agg([
        pl.len().alias("num_transactions"),
        pl.col("AMOUNT").sum().alias("total_amount"),
        pl.col("AMOUNT").mean().alias("avg_amount")
    ])
    .filter(pl.col("total_amount") > 500)
    .sort(["total_amount", "avg_amount"], descending=[True, True])
    .collect(engine = engine_multi)
)


/home/allisond/miniconda3/envs/rapids-25.06/lib/python3.10/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 54.94 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


CPU times: user 205 ms, sys: 487 ms, total: 692 ms
Wall time: 1.05 s


# Save the result back to DuckDB database

In [9]:
conn.execute("CREATE OR REPLACE TABLE result AS SELECT * FROM result")

# Close the databse connection

In [10]:
conn.close()